# Google Play Store Analytics Internship Project

## Task 1: Bubble Chart Analysis of App Size, Rating, and Installs

### Objective

The objective of this task is to analyze the relationship between application size (in MB) and average rating using an interactive Plotly bubble chart. The bubble size represents the number of installs.

The visualization will be created after cleaning, merging, and filtering the datasets according to the internship requirements.

### Datasets Used

- Play Store Data.csv
- User Reviews.csv

### Tools and Technologies

- Python
- Pandas
- NumPy
- Plotly
- VS Code (Jupyter Notebook)

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Interactive visualization
import plotly.express as px

# Date and time handling
from datetime import datetime
import pytz

In [2]:
%pip install plotly

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================================
# Load the Datasets
# ============================================================

playstore_df = pd.read_csv("Play Store Data.csv")
reviews_df = pd.read_csv("User Reviews.csv")

print("Datasets loaded successfully!")


Datasets loaded successfully!


In [4]:
# ============================================================
# Preview the Datasets
# ============================================================

print("Play Store Dataset") 
display(playstore_df.head())

print("\nUser Reviews Dataset")
display(reviews_df.head())


Play Store Dataset


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up



User Reviews Dataset


,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [5]:
# ============================================================
# Inspect Dataset Structure
# ============================================================

print("Play Store Dataset Information")
playstore_df.info()

print("\n" + "=" * 60 + "\n")

print("User Reviews Dataset Information")
reviews_df.info()

Play Store Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


User Reviews Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64295 entries, 0 to 64294
Data columns (total 5

In [6]:
# ============================================================
# Inspect the Reviews Column
# ============================================================

print("Data Type:", playstore_df["Reviews"].dtype)

print("\nFirst 10 values:")
print(playstore_df["Reviews"].head(10))

Data Type: object

First 10 values:
0       159
1       967
2     87510
3    215644
4       967
5       167
6       178
7     36815
8     13791
9       121
Name: Reviews, dtype: object


In [7]:
# ============================================================
# Remove the Corrupted Row Safely
# ============================================================

playstore_df = playstore_df.drop(index=10472, errors="ignore")
playstore_df = playstore_df.reset_index(drop=True)

print("Corrupted row handled successfully.")
print("Current dataset shape:", playstore_df.shape)

Corrupted row handled successfully.
Current dataset shape: (10840, 13)


In [8]:
# ============================================================
# Clean the Reviews Column
# ============================================================

playstore_df["Reviews"] = pd.to_numeric(playstore_df["Reviews"], errors="coerce")

print("Reviews Data Type:", playstore_df["Reviews"].dtype)
print("Missing values in Reviews:", playstore_df["Reviews"].isna().sum())

Reviews Data Type: int64
Missing values in Reviews: 0


In [9]:
# ============================================================
# Clean the Size Column
# ============================================================

def convert_size_to_mb(size):
    """
    Convert app size values into MB.
    Examples:
    '19M' -> 19.0
    '8.7M' -> 8.7
    '500k' -> 0.5
    'Varies with device' -> NaN
    """
    if pd.isna(size):
        return np.nan
    
    size = str(size).strip()
    
    if size == "Varies with device":
        return np.nan
    
    if size.endswith("M"):
        return float(size.replace("M", ""))
    
    if size.endswith("k"):
        return float(size.replace("k", "")) / 1000
    
    return np.nan


# Create a cleaned numeric size column in MB
playstore_df["Size_MB"] = playstore_df["Size"].apply(convert_size_to_mb)

# Verify the result
print("Size_MB Data Type:", playstore_df["Size_MB"].dtype)
print("Missing values in Size_MB:", playstore_df["Size_MB"].isna().sum())

playstore_df[["App", "Size", "Size_MB"]].head()

Size_MB Data Type: float64
Missing values in Size_MB: 1695


,App,Size,Size_MB
0,Photo Editor & Candy Camera & Grid & ScrapBook,19M,19.0
1,Coloring book moana,14M,14.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",8.7M,8.7
3,Sketch - Draw & Paint,25M,25.0
4,Pixel Draw - Number Art Coloring Book,2.8M,2.8


In [10]:


playstore_df["Installs"] = (
    playstore_df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

playstore_df["Installs"] = pd.to_numeric(playstore_df["Installs"], errors="coerce")

print("Installs Data Type:", playstore_df["Installs"].dtype)
print("Missing values in Installs:", playstore_df["Installs"].isna().sum())

playstore_df[["App", "Installs"]].head()

Installs Data Type: int64
Missing values in Installs: 0


,App,Installs
0,Photo Editor & Candy Camera & Grid & ScrapBook,10000
1,Coloring book moana,500000
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",5000000
3,Sketch - Draw & Paint,50000000
4,Pixel Draw - Number Art Coloring Book,100000


In [11]:
display(
    playstore_df[["App", "Installs"]]
    .head()
    .style.set_properties(**{
        "text-align": "left"
    })
)

,App,Installs
0,Photo Editor & Candy Camera & Grid & ScrapBook,10000
1,Coloring book moana,500000
2,"U Launcher Lite – FREE Live Cool Themes, Hide Apps",5000000
3,Sketch - Draw & Paint,50000000
4,Pixel Draw - Number Art Coloring Book,100000


In [12]:


print("Missing Ratings (Before):", playstore_df["Rating"].isna().sum())
playstore_df = playstore_df.dropna(subset=["Rating"])
playstore_df = playstore_df.reset_index(drop=True)
print("Missing Ratings (After):", playstore_df["Rating"].isna().sum())
print("Current Dataset Shape:", playstore_df.shape)

Missing Ratings (Before): 1474
Missing Ratings (After): 0
Current Dataset Shape: (9366, 14)


In [13]:

print("Total Rows:", len(reviews_df))
print("Unique Apps:", reviews_df["App"].nunique())

Total Rows: 64295
Unique Apps: 1074


In [14]:
# ============================================================
# Prepare User Reviews Data for Merging
# ============================================================

reviews_summary_df = (
    reviews_df
    .groupby("App", as_index=False)["Sentiment_Subjectivity"]
    .mean()
)

print("Reviews Summary Shape:", reviews_summary_df.shape)
reviews_summary_df.head()

Reviews Summary Shape: (1074, 2)


,App,Sentiment_Subjectivity
0,10 Best Foods for You,0.495455
1,104 找工作 - 找工作 找打工 找兼職 履歷健檢 履歷診療室,0.545516
2,11st,0.443957
3,1800 Contacts - Lens Store,0.591098
4,1LINE – One Line with One Touch,0.557315


In [15]:
# ============================================================
# Display Settings
# ============================================================

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 40)
pd.set_option("display.expand_frame_repr", False)

In [16]:
# ============================================================
# Merge Play Store Data with User Reviews Summary
# ============================================================

merged_df = pd.merge(
    playstore_df,
    reviews_summary_df,
    on="App",
    how="inner"
)

print("Merged Dataset Shape:", merged_df.shape)
print("Columns in Merged Dataset:")
print(merged_df.columns.tolist())

Merged Dataset Shape: (1531, 15)
Columns in Merged Dataset:
['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver', 'Size_MB', 'Sentiment_Subjectivity']


In [17]:
# ============================================================
# Apply Task 1 Filters
# ============================================================

selected_categories = [
    "GAME",
    "BEAUTY",
    "BUSINESS",
    "COMICS",
    "COMMUNICATION",
    "DATING",
    "ENTERTAINMENT",
    "SOCIAL",
    "EVENTS"
]

task1_df = merged_df[
    (merged_df["Rating"] > 3.5) &
    (merged_df["Reviews"] > 500) &
    (merged_df["Installs"] > 50000) &
    (merged_df["Category"].isin(selected_categories)) &
    (~merged_df["App"].str.contains("s", case=False, na=False)) &
    (merged_df["Sentiment_Subjectivity"] > 0.5) &
    (merged_df["Size_MB"].notna())
].copy()

print("Filtered Dataset Shape:", task1_df.shape)

task1_df[["App", "Category", "Rating", "Reviews", "Size_MB", "Installs", "Sentiment_Subjectivity"]].head()

Filtered Dataset Shape: (43, 15)


,App,Category,Rating,Reviews,Size_MB,Installs,Sentiment_Subjectivity
55,Google Primer,BUSINESS,4.4,62272,18.000,10000000,0.675000
58,Call Blocker,BUSINESS,4.6,188841,3.200,5000000,0.655431
134,Call Blocker,COMMUNICATION,4.1,17529,10.000,1000000,0.655431
135,"CallApp: Caller ID, Blocker & Phone ...",COMMUNICATION,4.4,483565,20.000,10000000,0.506481
140,Caller ID +,COMMUNICATION,4.0,9498,0.118,1000000,0.600000


In [18]:
# ============================================================
# Translate Required Category Names
# ============================================================

task1_df["Category"] = task1_df["Category"].replace({
    "BEAUTY": "सौंदर्य",          # Hindi
    "BUSINESS": "வணிகம்",        # Tamil
    "DATING": "Dating"           # We'll replace this next
})

# German translation for Dating
task1_df["Category"] = task1_df["Category"].replace({
    "Dating": "Partnersuche"
})

print("Category labels translated successfully.")

task1_df["Category"].value_counts()

Category labels translated successfully.


Category
GAME             25
Partnersuche      9
COMMUNICATION     4
வணிகம்            2
ENTERTAINMENT     2
SOCIAL            1
Name: count, dtype: int64

In [19]:
from IPython.display import HTML

display(HTML("""
<style>
table.dataframe td {
    text-align: left !important;
}
table.dataframe th {
    text-align: left !important;
}
</style>
"""))

In [59]:
task1_df["Category"].value_counts().reset_index(name="Count")
color_map = {
    "GAME": "pink"
}

In [60]:
# ============================================================
# STEP 8: CREATE PLOTLY BUBBLE CHART
# ============================================================

# Color mapping (only GAME is forced to Pink)
color_map = {
    "GAME": "pink"
}

fig = px.scatter(
    task1_df,
    x="Size_MB",
    y="Rating",
    size="Installs",
    color="Category",
    color_discrete_map=color_map,
    hover_name="App",
    hover_data={
        "Reviews": True,
        "Installs": ":,",
        "Sentiment_Subjectivity": ":.2f",
        "Size_MB": ":.2f"
    },
    title="Bubble Chart Analysis of App Size, Rating and Installs",
    labels={
        "Size_MB": "App Size (MB)",
        "Rating": "Average Rating",
        "Category": "Category",
        "Installs": "Number of Installs"
    },
    size_max=50
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

In [61]:
fig.update_traces(
    marker=dict(
        opacity=0.8,
        line=dict(width=1, color="black")
    )
)

In [62]:
# ============================================================
# STEP 9: DISPLAY CHART ONLY BETWEEN 5 PM AND 7 PM IST
# ============================================================

ist_timezone = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist_timezone).time()

start_time = datetime.strptime("17:00", "%H:%M").time()
end_time = datetime.strptime("19:00", "%H:%M").time()

if start_time <= current_time <= end_time:
    fig.show()
else:
    print("=" * 60)
    print("Dashboard Unavailable")
    print("=" * 60)
    print("The Bubble Chart is available only between 5:00 PM and 7:00 PM IST.")
    print("Current time is outside the permitted viewing window.")

Dashboard Unavailable
The Bubble Chart is available only between 5:00 PM and 7:00 PM IST.
Current time is outside the permitted viewing window.


# Task 2: Interactive Choropleth Map - Global Installs by Category

This task creates an interactive Plotly Choropleth map to visualize installs by app category.

Requirements:
- Show only the top 5 app categories.
- Exclude categories starting with A, C, G, or S.
- Highlight categories where installs exceed 1 million.
- Display the chart only between 6 PM IST and 8 PM IST.

In [63]:
# ============================================================
# Task 2 - Import Required Libraries
# ============================================================

import plotly.express as px
from datetime import datetime
import pytz

In [64]:
# Check available DataFrames in the notebook
import pandas as pd
[df_name for df_name in globals() if isinstance(globals()[df_name], pd.DataFrame)]

['__',
 '___',
 'playstore_df',
 'reviews_df',
 '_9',
 '_10',
 'reviews_summary_df',
 '_14',
 'merged_df',
 'task1_df',
 '_17',
 '_21',
 '_27',
 'task2_df',
 'category_installs',
 'top_5_categories',
 '_28',
 '_29',
 'choropleth_df',
 '_30',
 'task3_df',
 'task3_filtered',
 '_34',
 '_35',
 'monthly_installs',
 '_36',
 '_37',
 '_38',
 'category_data',
 'task4_df',
 '_40',
 'task4_filtered',
 '_41',
 '_42',
 'monthly_installs_task4',
 '_43',
 '_44',
 '_45',
 'highlight_points',
 'task5_df',
 'task5_filtered',
 'task5_summary',
 'top10_categories',
 '_51',
 'task6_df',
 '_52',
 '_53',
 'task6_filtered',
 '_54',
 'task6_top3',
 '_56',
 'task6_summary']

In [65]:
# Check columns needed for Task 2
playstore_df[['Category', 'Installs']].head()

,Category,Installs
0,ART_AND_DESIGN,10000
1,ART_AND_DESIGN,500000
2,ART_AND_DESIGN,5000000
3,ART_AND_DESIGN,50000000
4,ART_AND_DESIGN,100000


In [66]:
# ============================================================
# Task 2 - Filter Data for Top 5 Valid Categories
# ============================================================

# Create a safe copy for Task 2
task2_df = playstore_df.copy()

# Remove categories starting with A, C, G, or S
excluded_starting_letters = ('A', 'C', 'G', 'S')

task2_df = task2_df[
    ~task2_df['Category'].str.startswith(excluded_starting_letters, na=False)
]

# Calculate total installs by category
category_installs = (
    task2_df
    .groupby('Category', as_index=False)['Installs']
    .sum()
    .sort_values(by='Installs', ascending=False)
)

# Keep only the top 5 categories
top_5_categories = category_installs.head(5).copy()

top_5_categories

,Category,Installs
20,PRODUCTIVITY,14176070180
21,TOOLS,11450724500
7,FAMILY,10257701590
19,PHOTOGRAPHY,10088243130
16,NEWS_AND_MAGAZINES,7496210650


In [67]:
# ============================================================
# Task 2 - Identify Categories with More Than 1 Million Installs
# ============================================================

top_5_categories['Highlight'] = top_5_categories['Installs'].apply(
    lambda installs: "Above 1 Million" if installs > 1_000_000 else "Below 1 Million"
)

top_5_categories

,Category,Installs,Highlight
20,PRODUCTIVITY,14176070180,Above 1 Million
21,TOOLS,11450724500,Above 1 Million
7,FAMILY,10257701590,Above 1 Million
19,PHOTOGRAPHY,10088243130,Above 1 Million
16,NEWS_AND_MAGAZINES,7496210650,Above 1 Million


In [68]:
# ============================================================
# Task 2 - Prepare Data for Choropleth Map
# ============================================================

# Create a copy to avoid modifying the original top_5_categories DataFrame
choropleth_df = top_5_categories.copy()

# Assign sample countries for choropleth visualization
choropleth_df['Country'] = ['India', 'United States', 'Brazil', 'Germany', 'Canada']

# Add ISO country codes required by Plotly choropleth
choropleth_df['ISO_Code'] = ['IND', 'USA', 'BRA', 'DEU', 'CAN']

# Display prepared map data
choropleth_df

,Category,Installs,Highlight,Country,ISO_Code
20,PRODUCTIVITY,14176070180,Above 1 Million,India,IND
21,TOOLS,11450724500,Above 1 Million,United States,USA
7,FAMILY,10257701590,Above 1 Million,Brazil,BRA
19,PHOTOGRAPHY,10088243130,Above 1 Million,Germany,DEU
16,NEWS_AND_MAGAZINES,7496210650,Above 1 Million,Canada,CAN


In [69]:
# ============================================================
# Task 2 - Final Interactive Choropleth Map
# ============================================================

# Get current IST time
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

# Allowed display time: 6 PM to 8 PM IST
start_time = datetime.strptime("18:00", "%H:%M").time()
end_time = datetime.strptime("20:00", "%H:%M").time()

if start_time <= current_time <= end_time:

    fig = px.choropleth(
        choropleth_df,
        locations='ISO_Code',
        color='Installs',
        hover_name='Country',
        hover_data={
            'Category': True,
            'Installs': ':,',
            'Highlight': True,
            'ISO_Code': False
        },
        color_continuous_scale='Plasma',
        title='Global Installs by Top 5 Filtered Google Play Categories'
    )

    fig.update_geos(
        projection_type='natural earth',
        showcoastlines=True,
        coastlinecolor='Black',
        showland=True,
        landcolor='lightgray',
        showocean=True,
        oceancolor='lightblue'
    )

    fig.update_layout(
        title_x=0.5,
        height=600,
        margin=dict(l=20, r=20, t=60, b=20)
    )

    fig.show()

else:
    print(
        "Dashboard Notice: This choropleth map is hidden because it is "
        "outside the allowed display time: 6:00 PM to 8:00 PM IST."
    )

Dashboard Notice: This choropleth map is hidden because it is outside the allowed display time: 6:00 PM to 8:00 PM IST.


## Task 2 Note

The original dataset does not contain a country or geographic location column.  
Since a Choropleth map requires geographic data, sample countries and ISO country codes were assigned to the top 5 filtered categories for visualization purposes.

The chart is displayed only between 6 PM IST and 8 PM IST as required.


# Task 3: Time Series Line Chart - Total Installs Trend by Category

In [70]:
# ============================================================
# Task 3 - Time Series Line Chart: Installs Trend by Category
# ============================================================

import pandas as pd
import plotly.express as px
from datetime import datetime
import pytz

In [71]:
# Create a copy of the Play Store dataset
task3_df = playstore_df.copy()

# Display the columns in the dataset
print("Columns in the dataset:\n")
print(task3_df.columns.tolist())

Columns in the dataset:

['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver', 'Size_MB']


In [72]:
# ============================================================
# Filter the dataset according to Task 3 requirements
# ============================================================

# Keep only apps with Reviews greater than 500
task3_filtered = task3_df[task3_df["Reviews"] > 500]

# Keep only categories starting with E, C, or B
task3_filtered = task3_filtered[
    task3_filtered["Category"].str.startswith(("E", "C", "B"), na=False)
]

# Remove apps whose names start with X, Y, or Z
task3_filtered = task3_filtered[
    ~task3_filtered["App"].str.startswith(("X", "Y", "Z"), na=False)
]

# Remove apps containing the letter 'S' or 's'
task3_filtered = task3_filtered[
    ~task3_filtered["App"].str.contains("S", case=False, na=False)
]

# Display the number of remaining records
print("Remaining records:", len(task3_filtered))

# Display the first five rows
task3_filtered.head()

Remaining records: 244


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Size_MB
106,Ulta Beauty,BEAUTY,4.7,42050,Varies with device,1000000,Free,0,Everyone,Beauty,"June 5, 2018",5.4,5.0 and up,NaN
129,Rainbow Camera,BEAUTY,3.7,3871,23M,1000000,Free,0,Everyone,Beauty,"July 30, 2018",2.2.0,4.0 and up,23.0
133,E-Book Read - Read Book for free,BOOKS_AND_REFERENCE,4.5,1857,4.9M,50000,Free,0,Everyone,Books & Reference,"August 3, 2018",1.3.2,4.4 and up,4.9
134,Download free book with green book,BOOKS_AND_REFERENCE,4.6,4478,9.5M,100000,Free,0,Everyone 10+,Books & Reference,"July 31, 2017",1.1,4.0 and up,9.5
135,Wikipedia,BOOKS_AND_REFERENCE,4.4,577550,Varies with device,10000000,Free,0,Everyone,Books & Reference,"August 2, 2018",Varies with device,Varies with device,NaN


In [73]:
# ============================================================
# Clean Installs and Last Updated columns
# ============================================================

# Make a copy to avoid warnings
task3_filtered = task3_filtered.copy()

# Convert Installs from text to number
# Example: "10,000+" becomes 10000
task3_filtered["Installs"] = (
    task3_filtered["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

task3_filtered["Installs"] = pd.to_numeric(task3_filtered["Installs"], errors="coerce")

# Convert Last Updated to datetime format
task3_filtered["Last Updated"] = pd.to_datetime(
    task3_filtered["Last Updated"],
    errors="coerce"
)

# Create a Month column for monthly time-series analysis
task3_filtered["Month"] = task3_filtered["Last Updated"].dt.to_period("M").dt.to_timestamp()

# Remove rows where important converted values are missing
task3_filtered = task3_filtered.dropna(subset=["Installs", "Last Updated", "Month"])

# Display sample cleaned data
task3_filtered[["App", "Category", "Reviews", "Installs", "Last Updated", "Month"]].head()

,App,Category,Reviews,Installs,Last Updated,Month
106,Ulta Beauty,BEAUTY,42050,1000000,2018-06-05,2018-06-01
129,Rainbow Camera,BEAUTY,3871,1000000,2018-07-30,2018-07-01
133,E-Book Read - Read Book for free,BOOKS_AND_REFERENCE,1857,50000,2018-08-03,2018-08-01
134,Download free book with green book,BOOKS_AND_REFERENCE,4478,100000,2017-07-31,2017-07-01
135,Wikipedia,BOOKS_AND_REFERENCE,577550,10000000,2018-08-02,2018-08-01


In [74]:
# ============================================================
# Calculate total installs for each category by month
# ============================================================

monthly_installs = (
    task3_filtered
    .groupby(["Month", "Category"], as_index=False)["Installs"]
    .sum()
)

# Display the first few rows
monthly_installs.head()

,Month,Category,Installs
0,2014-01-01,COMMUNICATION,100000
1,2014-03-01,COMMUNICATION,100000
2,2014-05-01,BUSINESS,100000
3,2014-07-01,COMMUNICATION,10000000
4,2014-07-01,EDUCATION,10000


In [75]:
# ============================================================
# Calculate Month-over-Month Growth
# ============================================================

# Sort values before calculating growth
monthly_installs = monthly_installs.sort_values(
    by=["Category", "Month"]
)

# Calculate percentage growth month-over-month
monthly_installs["Growth"] = (
    monthly_installs
    .groupby("Category")["Installs"]
    .pct_change()
)

# Identify significant growth (>20%)
monthly_installs["Significant_Growth"] = (
    monthly_installs["Growth"] > 0.20
)

# Display the results
monthly_installs.head(10)

,Month,Category,Installs,Growth,Significant_Growth
46,2018-03-01,BEAUTY,5000,NaN,False
58,2018-06-01,BEAUTY,1000000,199.000000,True
64,2018-07-01,BEAUTY,1000000,0.000000,False
72,2018-08-01,BEAUTY,100000,-0.900000,False
5,2014-10-01,BOOKS_AND_REFERENCE,500000,NaN,False
6,2014-11-01,BOOKS_AND_REFERENCE,5000000,9.000000,True
10,2015-07-01,BOOKS_AND_REFERENCE,10000000,1.000000,True
18,2016-06-01,BOOKS_AND_REFERENCE,60000,-0.994000,False
21,2016-08-01,BOOKS_AND_REFERENCE,100000,0.666667,True
27,2017-02-01,BOOKS_AND_REFERENCE,100000,0.000000,False


In [76]:
# ============================================================
# Translate selected category names for display on graph
# ============================================================

category_translation = {
    "BEAUTY": "सौंदर्य",      # Beauty in Hindi
    "BUSINESS": "வணிகம்",    # Business in Tamil
    "DATING": "Dating"        # Dating in German
}

monthly_installs["Category_Display"] = monthly_installs["Category"].replace(category_translation)

# Display sample translated categories
monthly_installs[["Category", "Category_Display"]].drop_duplicates().head(10)

,Category,Category_Display
46,BEAUTY,सौंदर्य
5,BOOKS_AND_REFERENCE,BOOKS_AND_REFERENCE
2,BUSINESS,வணிகம்
28,COMICS,COMICS
0,COMMUNICATION,COMMUNICATION
4,EDUCATION,EDUCATION
14,ENTERTAINMENT,ENTERTAINMENT
57,EVENTS,EVENTS


In [77]:
# ============================================================
# Task 3 - Professional Time Series Line Chart
# ============================================================

import plotly.graph_objects as go

fig = go.Figure()

colors = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#17becf"
]

categories = monthly_installs["Category_Display"].unique()

for i, category in enumerate(categories):

    category_data = (
        monthly_installs[
            monthly_installs["Category_Display"] == category
        ]
        .sort_values("Month")
        .reset_index(drop=True)
    )

    # ---------------- Main Line ----------------

    fig.add_trace(
        go.Scatter(
            x=category_data["Month"],
            y=category_data["Installs"],
            mode="lines+markers",
            name=category,
            line=dict(
                color=colors[i % len(colors)],
                width=2
            ),
            marker=dict(size=6),
            customdata=category_data["Growth"],
            hovertemplate=
            "<b>%{fullData.name}</b><br>"
            "Month: %{x|%b %Y}<br>"
            "Installs: %{y:,}<br>"
            "Growth: %{customdata:.2%}<extra></extra>"
        )
    )

    # -------- Highlight only significant growth segments --------

    for j in range(1, len(category_data)):

        if category_data.loc[j, "Significant_Growth"]:

            fig.add_trace(

                go.Scatter(

                    x=[
                        category_data.loc[j-1, "Month"],
                        category_data.loc[j, "Month"]
                    ],

                    y=[
                        category_data.loc[j-1, "Installs"],
                        category_data.loc[j, "Installs"]
                    ],

                    mode="lines",

                    line=dict(
                        color=colors[i % len(colors)],
                        width=10
                    ),

                    opacity=0.25,

                    showlegend=False,

                    hoverinfo="skip"

                )
            )

fig.update_layout(

    title="Trend of Total Installs Over Time by App Category",

    xaxis_title="Month",

    yaxis_title="Total Installs",

    template="plotly_white",

    hovermode="x unified",

    legend_title="App Category",

    height=650

)

fig.update_yaxes(type="log")

fig.show()
# ============================================================
# Display chart only between 6 PM and 9 PM IST
# ============================================================

ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

start_time = datetime.strptime("18:00", "%H:%M").time()
end_time = datetime.strptime("21:00", "%H:%M").time()

if start_time <= current_time <= end_time:
    fig.show()
else:
    print("The Task 3 dashboard is available only between 6:00 PM and 9:00 PM IST.")

# Task 4: Stacked Area Chart - Cumulative Installs Over Time by Category

In [78]:
# ============================================================
# Task 4 - Stacked Area Chart: Cumulative Installs by Category
# ============================================================

# Create a separate copy for Task 4
task4_df = playstore_df.copy()

# Display required columns for verification
task4_df[["App", "Category", "Rating", "Reviews", "Size_MB", "Installs", "Last Updated"]].head()

,App,Category,Rating,Reviews,Size_MB,Installs,Last Updated
0,Photo Editor & Candy Camera & Grid &...,ART_AND_DESIGN,4.1,159,19.0,10000,"January 7, 2018"
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14.0,500000,"January 15, 2018"
2,U Launcher Lite – FREE Live Cool The...,ART_AND_DESIGN,4.7,87510,8.7,5000000,"August 1, 2018"
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25.0,50000000,"June 8, 2018"
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8,100000,"June 20, 2018"


In [79]:
# ============================================================
# Apply Task 4 filters
# ============================================================

task4_filtered = task4_df.copy()

task4_filtered = task4_filtered[
    (task4_filtered["Rating"] >= 4.2) &
    (~task4_filtered["App"].str.contains(r"\d", regex=True, na=False)) &
    (task4_filtered["Category"].str.startswith(("T", "P"), na=False)) &
    (task4_filtered["Reviews"] > 1000) &
    (task4_filtered["Size_MB"] >= 20) &
    (task4_filtered["Size_MB"] <= 80)
]

print("Remaining records:", len(task4_filtered))

task4_filtered[["App", "Category", "Rating", "Reviews", "Size_MB", "Installs", "Last Updated"]].head()

Remaining records: 139


,App,Category,Rating,Reviews,Size_MB,Installs,Last Updated
2663,"Shutterfly: Free Prints, Photo Books...",PHOTOGRAPHY,4.6,98716,59.0,5000000,"August 1, 2018"
2664,FreePrints – Free Photos Delivered,PHOTOGRAPHY,4.8,109500,37.0,1000000,"August 2, 2018"
2672,"Face Filter, Selfie Editor - Sweet C...",PHOTOGRAPHY,4.7,142634,22.0,10000000,"July 6, 2018"
2683,Makeup Editor -Beauty Photo Editor &...,PHOTOGRAPHY,4.5,3378,30.0,1000000,"July 25, 2018"
2684,Makeup Photo Editor: Makeup Camera &...,PHOTOGRAPHY,4.4,10525,25.0,1000000,"July 27, 2018"


In [80]:
# ============================================================
# Clean Installs and Last Updated columns
# ============================================================

# Convert Installs to numeric
task4_filtered["Installs"] = (
    task4_filtered["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

task4_filtered["Installs"] = pd.to_numeric(
    task4_filtered["Installs"],
    errors="coerce"
)

# Convert Last Updated to datetime
task4_filtered["Last Updated"] = pd.to_datetime(
    task4_filtered["Last Updated"],
    errors="coerce"
)

# Create Month column
task4_filtered["Month"] = (
    task4_filtered["Last Updated"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Remove rows with missing values
task4_filtered = task4_filtered.dropna(
    subset=["Installs", "Last Updated", "Month"]
)

# Display cleaned data
task4_filtered[
    ["App", "Category", "Installs", "Last Updated", "Month"]
].head()

,App,Category,Installs,Last Updated,Month
2663,"Shutterfly: Free Prints, Photo Books...",PHOTOGRAPHY,5000000,2018-08-01,2018-08-01
2664,FreePrints – Free Photos Delivered,PHOTOGRAPHY,1000000,2018-08-02,2018-08-01
2672,"Face Filter, Selfie Editor - Sweet C...",PHOTOGRAPHY,10000000,2018-07-06,2018-07-01
2683,Makeup Editor -Beauty Photo Editor &...,PHOTOGRAPHY,1000000,2018-07-25,2018-07-01
2684,Makeup Photo Editor: Makeup Camera &...,PHOTOGRAPHY,1000000,2018-07-27,2018-07-01


In [81]:
# ============================================================
# Calculate Monthly and Cumulative Installs
# ============================================================

# Total installs by Month and Category
monthly_installs_task4 = (
    task4_filtered
    .groupby(["Month", "Category"], as_index=False)["Installs"]
    .sum()
)

# Sort data
monthly_installs_task4 = monthly_installs_task4.sort_values(
    ["Category", "Month"]
)

# Calculate cumulative installs
monthly_installs_task4["Cumulative_Installs"] = (
    monthly_installs_task4
    .groupby("Category")["Installs"]
    .cumsum()
)

# Display sample
monthly_installs_task4.head(10)

,Month,Category,Installs,Cumulative_Installs
16,2018-03-01,PARENTING,100000,100000
19,2018-05-01,PARENTING,10000000,10100000
26,2018-07-01,PARENTING,600000,10700000
2,2016-12-01,PERSONALIZATION,2000000,2000000
7,2017-09-01,PERSONALIZATION,1000000,3000000
11,2018-01-01,PERSONALIZATION,10000000,13000000
14,2018-02-01,PERSONALIZATION,10000000,23000000
22,2018-06-01,PERSONALIZATION,10000000,33000000
27,2018-07-01,PERSONALIZATION,11500000,44500000
32,2018-08-01,PERSONALIZATION,20000000,64500000


In [82]:
# ============================================================
# Calculate Month-over-Month Growth
# ============================================================

# Calculate percentage growth for each category
monthly_installs_task4["Growth"] = (
    monthly_installs_task4
    .groupby("Category")["Cumulative_Installs"]
    .pct_change()
)

# Identify months with growth greater than 25%
monthly_installs_task4["Highlight"] = (
    monthly_installs_task4["Growth"] > 0.25
)

# Display the results
monthly_installs_task4.head(10)

,Month,Category,Installs,Cumulative_Installs,Growth,Highlight
16,2018-03-01,PARENTING,100000,100000,NaN,False
19,2018-05-01,PARENTING,10000000,10100000,100.000000,True
26,2018-07-01,PARENTING,600000,10700000,0.059406,False
2,2016-12-01,PERSONALIZATION,2000000,2000000,NaN,False
7,2017-09-01,PERSONALIZATION,1000000,3000000,0.500000,True
11,2018-01-01,PERSONALIZATION,10000000,13000000,3.333333,True
14,2018-02-01,PERSONALIZATION,10000000,23000000,0.769231,True
22,2018-06-01,PERSONALIZATION,10000000,33000000,0.434783,True
27,2018-07-01,PERSONALIZATION,11500000,44500000,0.348485,True
32,2018-08-01,PERSONALIZATION,20000000,64500000,0.449438,True


In [83]:
# ============================================================
# Translate categories for chart legend
# ============================================================

category_translation_task4 = {
    "TRAVEL_AND_LOCAL": "Voyages et Local",   # French
    "PRODUCTIVITY": "Productividad",          # Spanish
    "PHOTOGRAPHY": "写真"                     # Japanese
}

monthly_installs_task4["Category_Display"] = (
    monthly_installs_task4["Category"]
    .replace(category_translation_task4)
)

# Display translated categories
monthly_installs_task4[
    ["Category", "Category_Display"]
].drop_duplicates()

,Category,Category_Display
16,PARENTING,PARENTING
2,PERSONALIZATION,PERSONALIZATION
0,PHOTOGRAPHY,写真
1,PRODUCTIVITY,Productividad
6,TOOLS,TOOLS
9,TRAVEL_AND_LOCAL,Voyages et Local


In [84]:
# ============================================================
# Create Stacked Area Chart
# ============================================================

import plotly.express as px

fig = px.area(
    monthly_installs_task4,
    x="Month",
    y="Cumulative_Installs",
    color="Category_Display",
    title="Cumulative Installs Over Time by App Category",
    labels={
        "Month": "Month",
        "Cumulative_Installs": "Cumulative Installs",
        "Category_Display": "App Category"
    },
    template="plotly_white"
)

# Increase opacity for highlighted months
highlight_points = monthly_installs_task4[
    monthly_installs_task4["Highlight"]
]

fig.add_scatter(
    x=highlight_points["Month"],
    y=highlight_points["Cumulative_Installs"],
    mode="markers",
    marker=dict(
        size=12,
        color="black",
        opacity=0.8,
        symbol="circle"
    ),
    name="Growth > 25%"
)

fig.update_layout(
    hovermode="x unified",
    height=650,
    legend_title="App Category"
)


# ============================================================
# Display chart only between 4 PM and 6 PM IST
# ============================================================

ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

start_time = datetime.strptime("16:00", "%H:%M").time()
end_time = datetime.strptime("18:00", "%H:%M").time()

if start_time <= current_time <= end_time:
    fig.show()
else:
    print("The Task 4 dashboard is available only between 4:00 PM and 6:00 PM IST.")

The Task 4 dashboard is available only between 4:00 PM and 6:00 PM IST.


# Task 5: Grouped Bar Chart - Average Rating and Total Reviews for Top 10 Categories

In [85]:
# ============================================================
# Task 5 - Data Preparation
# ============================================================

# Create a copy so previous tasks are unaffected
task5_df = playstore_df.copy()

# Convert Reviews to numeric
task5_df["Reviews"] = pd.to_numeric(task5_df["Reviews"], errors="coerce")

# Convert Rating to numeric
task5_df["Rating"] = pd.to_numeric(task5_df["Rating"], errors="coerce")

# Clean Installs column safely
# Works whether Installs is text like "1,000+" or already numeric
task5_df["Installs"] = (
    task5_df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)

task5_df["Installs"] = pd.to_numeric(task5_df["Installs"], errors="coerce")

# Clean Size column safely
# Convert all sizes into MB
task5_df["Size"] = task5_df["Size"].astype(str)

task5_df["Size"] = task5_df["Size"].replace("Varies with device", pd.NA)

task5_df.loc[task5_df["Size"].str.contains("M", na=False), "Size"] = (
    task5_df.loc[task5_df["Size"].str.contains("M", na=False), "Size"]
    .str.replace("M", "", regex=False)
)

task5_df.loc[task5_df["Size"].str.contains("k", na=False), "Size"] = (
    task5_df.loc[task5_df["Size"].str.contains("k", na=False), "Size"]
    .str.replace("k", "", regex=False)
    .astype(float) / 1000
)

task5_df["Size"] = pd.to_numeric(task5_df["Size"], errors="coerce")

# Convert Last Updated to datetime
task5_df["Last Updated"] = pd.to_datetime(
    task5_df["Last Updated"],
    errors="coerce"
)

print("Task 5 dataset prepared successfully!")

print("\nDataset Shape:")
print(task5_df.shape)

print("\nData Types:")
print(task5_df[
    ["Category", "Rating", "Reviews", "Installs", "Size", "Last Updated"]
].dtypes)

Task 5 dataset prepared successfully!

Dataset Shape:
(9366, 14)

Data Types:
Category                object
Rating                 float64
Reviews                  int64
Installs                 int64
Size                   float64
Last Updated    datetime64[ns]
dtype: object


In [86]:
# ============================================================
# Task 5 - Apply Required Filters
# ============================================================

task5_filtered = task5_df[
    (task5_df["Rating"] >= 4.0) &
    (task5_df["Size"] < 10) &
    (task5_df["Last Updated"].dt.month == 1)
].copy()

# Remove rows with missing values in important columns
task5_filtered = task5_filtered.dropna(
    subset=[
        "Category",
        "Rating",
        "Reviews",
        "Installs"
    ]
)

print("Filtered Dataset Shape:")
print(task5_filtered.shape)

print("\nFirst 5 Rows:")
display(
    task5_filtered[
        ["Category", "Rating", "Reviews", "Installs", "Size", "Last Updated"]
    ].head()
)

Filtered Dataset Shape:
(140, 14)

First 5 Rows:


,Category,Rating,Reviews,Installs,Size,Last Updated
143,BOOKS_AND_REFERENCE,4.6,1435,500000,2.8,2018-01-21
421,COMMUNICATION,4.4,27540,1000000,3.7,2018-01-05
441,COMMUNICATION,4.2,88427,5000000,5.1,2018-01-06
765,EDUCATION,4.0,2525,100000,1.2,2015-01-11
804,ENTERTAINMENT,4.0,11656,1000000,4.5,2018-01-20


In [87]:
# ============================================================
# Task 5 - Top 10 Categories by Installs
# ============================================================

task5_summary = (
    task5_filtered
    .groupby("Category", as_index=False)
    .agg({
        "Installs": "sum",
        "Rating": "mean",
        "Reviews": "sum"
    })
)

# Rename columns for better readability
task5_summary.rename(columns={
    "Installs": "Total Installs",
    "Rating": "Average Rating",
    "Reviews": "Total Reviews"
}, inplace=True)

# Select Top 10 categories based on installs
top10_categories = (
    task5_summary
    .sort_values(by="Total Installs", ascending=False)
    .head(10)
)

print("Top 10 Categories:")
display(top10_categories)

Top 10 Categories:


,Category,Total Installs,Average Rating,Total Reviews
22,TOOLS,24578000,4.371429,377047
9,GAME,22600000,4.114286,794665
7,FAMILY,16267100,4.427586,308236
16,PERSONALIZATION,12214500,4.300000,122821
2,BUSINESS,11005010,4.275000,192719
4,COMMUNICATION,7210100,4.385714,153777
18,PRODUCTIVITY,4070000,4.175000,82806
10,HEALTH_AND_FITNESS,2800000,4.266667,52962
21,SPORTS,1200000,4.400000,18180
17,PHOTOGRAPHY,1002200,4.600000,49283


In [88]:
# ============================================================
# Task 5 - Final Grouped Bar Chart with Time Restriction
# ============================================================

from datetime import datetime
import pytz
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Get current IST time
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

# TEMPORARY TESTING ONLY:
# Uncomment the next line to view the chart outside the allowed time.
# Comment it again before final submission.
# current_time = datetime.strptime("16:00", "%H:%M").time()

# Allowed display time: 3 PM to 5 PM IST
start_time = datetime.strptime("15:00", "%H:%M").time()
end_time = datetime.strptime("17:00", "%H:%M").time()

if start_time <= current_time <= end_time:

    # Create figure with secondary Y-axis
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Bar 1: Average Rating
    fig.add_trace(
        go.Bar(
            x=top10_categories["Category"],
            y=top10_categories["Average Rating"],
            name="Average Rating",
            marker_color="#4169E1"
        ),
        secondary_y=False
    )

    # Bar 2: Total Reviews
    fig.add_trace(
        go.Bar(
            x=top10_categories["Category"],
            y=top10_categories["Total Reviews"],
            name="Total Reviews",
            marker_color="#FF7F50"
        ),
        secondary_y=True
    )

    # Chart layout
    fig.update_layout(
        title="<b>Top 10 App Categories by Installs: Average Rating vs Total Reviews</b>",
        title_x=0.5,

        xaxis=dict(
            title="<b>App Category</b>",
            tickangle=-30
        ),

        barmode="group",
        template="plotly_white",

        height=650,
        width=1200,

        bargap=0.25,

        font=dict(
            size=15
        ),

        legend=dict(
            orientation="h",
            y=1.05,
            x=0.5,
            xanchor="center"
        ),

        margin=dict(
            t=110
        )
    )

    # Left Y-axis: Average Rating
    fig.update_yaxes(
        title_text="<b>Average Rating</b>",
        range=[4.0, 5.0],
        secondary_y=False
    )

    # Right Y-axis: Total Reviews
    fig.update_yaxes(
        title_text="<b>Total Reviews</b>",
        tickformat="~s",
        secondary_y=True
    )

    fig.show()

else:
    print("Task 5 chart is available only between 3:00 PM and 5:00 PM IST.")

Task 5 chart is available only between 3:00 PM and 5:00 PM IST.


# Task 6: Dual-Axis Chart

### Objective
Compare the **Average Installs** and **Revenue** of **Free** and **Paid** apps within the **Top 3 Categories** after applying the required filters.

**Filters Applied**
- Installs ≥ 10,000
- Revenue ≥ $10,000
- Android Version > 4.0
- Size > 15 MB
- Content Rating = Everyone
- App Name ≤ 30 characters

**Time Restriction**
- Display chart only between **1:00 PM and 2:00 PM IST**.

In [89]:
# ============================================================
# Task 6 - Step 1: Inspect Required Columns
# ============================================================

# Check available columns
print("Columns in Play Store dataset:")
print(playstore_df.columns.tolist())

# Preview important columns for Task 6
playstore_df[
    ['App', 'Category', 'Installs', 'Type', 'Price', 'Size', 'Content Rating', 'Android Ver']
].head()

Columns in Play Store dataset:
['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver', 'Size_MB']


,App,Category,Installs,Type,Price,Size,Content Rating,Android Ver
0,Photo Editor & Candy Camera & Grid &...,ART_AND_DESIGN,10000,Free,0,19M,Everyone,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,500000,Free,0,14M,Everyone,4.0.3 and up
2,U Launcher Lite – FREE Live Cool The...,ART_AND_DESIGN,5000000,Free,0,8.7M,Everyone,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,50000000,Free,0,25M,Teen,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,100000,Free,0,2.8M,Everyone,4.4 and up


## Step 2: Prepare Required Columns

Clean installs, price, size, Android version, and calculate revenue for filtering.

In [90]:
# ============================================================
# Task 6 - Step 2: Prepare Required Columns
# ============================================================

task6_df = playstore_df.copy()

# Clean Installs column
task6_df['Installs'] = (
    task6_df['Installs']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
)

task6_df['Installs'] = pd.to_numeric(task6_df['Installs'], errors='coerce')

# Clean Price column
task6_df['Price'] = (
    task6_df['Price']
    .astype(str)
    .str.replace('$', '', regex=False)
)

task6_df['Price'] = pd.to_numeric(task6_df['Price'], errors='coerce')

# Calculate Revenue
task6_df['Revenue'] = task6_df['Installs'] * task6_df['Price']

# Preview output
task6_df[['App', 'Category', 'Type', 'Installs', 'Price', 'Revenue']].head()

,App,Category,Type,Installs,Price,Revenue
0,Photo Editor & Candy Camera & Grid &...,ART_AND_DESIGN,Free,10000,0.0,0.0
1,Coloring book moana,ART_AND_DESIGN,Free,500000,0.0,0.0
2,U Launcher Lite – FREE Live Cool The...,ART_AND_DESIGN,Free,5000000,0.0,0.0
3,Sketch - Draw & Paint,ART_AND_DESIGN,Free,50000000,0.0,0.0
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,Free,100000,0.0,0.0


## Step 3: Clean Size and Android Version

Convert the **Size** and **Android Version** columns into numeric values so they can be used for filtering. Also calculate the app name length for the character limit requirement.

In [91]:
# ============================================================
# Task 6 - Step 3: Clean Size, Android Version and App Name
# ============================================================

# ----- Clean Size (convert to MB) -----
task6_df['Size_MB'] = (
    task6_df['Size']
    .astype(str)
    .str.replace('M', '', regex=False)
    .str.replace('k', '', regex=False)
    .str.replace('Varies with device', '', regex=False)
)

task6_df['Size_MB'] = pd.to_numeric(task6_df['Size_MB'], errors='coerce')

# ----- Clean Android Version -----
task6_df['Android_Version'] = (
    task6_df['Android Ver']
    .astype(str)
    .str.extract(r'(\d+\.\d+)')
)

task6_df['Android_Version'] = pd.to_numeric(
    task6_df['Android_Version'],
    errors='coerce'
)

# ----- App Name Length -----
task6_df['App_Name_Length'] = task6_df['App'].astype(str).str.len()

# Preview cleaned columns
task6_df[
    ['App', 'Size', 'Size_MB', 'Android Ver',
     'Android_Version', 'App_Name_Length']
].head()

,App,Size,Size_MB,Android Ver,Android_Version,App_Name_Length
0,Photo Editor & Candy Camera & Grid &...,19M,19.0,4.0.3 and up,4.0,46
1,Coloring book moana,14M,14.0,4.0.3 and up,4.0,19
2,U Launcher Lite – FREE Live Cool The...,8.7M,8.7,4.0.3 and up,4.0,50
3,Sketch - Draw & Paint,25M,25.0,4.2 and up,4.2,21
4,Pixel Draw - Number Art Coloring Book,2.8M,2.8,4.4 and up,4.4,37


In [92]:
# ============================================================
# Task 6 - Step 4: Apply Internship Filters
# ============================================================

task6_filtered = task6_df[
    (task6_df['Installs'] >= 10000) &
    (task6_df['Revenue'] >= 10000) &
    (task6_df['Android_Version'] > 4.0) &
    (task6_df['Size_MB'] > 15) &
    (task6_df['Content Rating'] == 'Everyone') &
    (task6_df['App_Name_Length'] <= 30)
].copy()

print("Total apps after applying Task 6 filters:", task6_filtered.shape[0])

task6_filtered[
    ['App', 'Category', 'Type', 'Installs', 'Price', 'Revenue',
     'Android_Version', 'Size_MB', 'Content Rating', 'App_Name_Length']
].head()

Total apps after applying Task 6 filters: 33


,App,Category,Type,Installs,Price,Revenue,Android_Version,Size_MB,Content Rating,App_Name_Length
801,Toca Life: City,EDUCATION,Paid,500000,3.99,1995000.0,4.4,24.0,Everyone,15
802,Toca Life: Hospital,EDUCATION,Paid,100000,3.99,399000.0,4.4,24.0,Everyone,19
1270,Meditation Studio,HEALTH_AND_FITNESS,Paid,10000,3.99,39900.0,4.3,29.0,Everyone,17
1749,The Game of Life,GAME,Paid,100000,2.99,299000.0,4.4,63.0,Everyone,16
1751,The Room: Old Sins,GAME,Paid,100000,4.99,499000.0,4.4,48.0,Everyone,18


In [93]:
# ============================================================
# Task 6 - Step 5: Verify Remaining App Types
# ============================================================

print("App Types after filtering:")
print(task6_filtered['Type'].value_counts())

App Types after filtering:
Type
Paid    33
Name: count, dtype: int64


## Step 5: Identify the Top 3 Categories

Find the top three app categories from the filtered dataset based on the total number of installs.

In [94]:
# ============================================================
# Task 6 - Step 5: Identify Top 3 Categories
# ============================================================

# Find top 3 categories based on total installs
top3_categories = (
    task6_filtered
    .groupby('Category')['Installs']
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

print("Top 3 Categories:")
print(top3_categories)

# Filter dataset to include only the top 3 categories
task6_top3 = task6_filtered[
    task6_filtered['Category'].isin(top3_categories.index)
].copy()

# Preview
task6_top3[['App', 'Category', 'Type', 'Installs', 'Revenue']].head()

Top 3 Categories:
Category
PHOTOGRAPHY        3000000
FAMILY             2670000
PERSONALIZATION    2010000
Name: Installs, dtype: int64


,App,Category,Type,Installs,Revenue
2068,Toca Life: City,FAMILY,Paid,500000,1995000.0
2744,Facetune - For Free,PHOTOGRAPHY,Paid,1000000,5990000.0
2773,Facetune - For Free,PHOTOGRAPHY,Paid,1000000,5990000.0
2811,Facetune - For Free,PHOTOGRAPHY,Paid,1000000,5990000.0
3264,HD Widgets,PERSONALIZATION,Paid,1000000,990000.0


## Step 6: Prepare Data for Visualization

Calculate the average installs and total revenue for each app type (Free/Paid) within the top 3 categories.

In [95]:
# ============================================================
# Task 6 - Step 6: Prepare Summary Data
# ============================================================

task6_summary = (
    task6_top3
    .groupby(['Category', 'Type'])
    .agg(
        Average_Installs=('Installs', 'mean'),
        Total_Revenue=('Revenue', 'sum')
    )
    .reset_index()
)

print(task6_summary)

          Category  Type  Average_Installs  Total_Revenue
0           FAMILY  Paid     381428.571429      5578300.0
1  PERSONALIZATION  Paid     670000.000000      1999900.0
2      PHOTOGRAPHY  Paid    1000000.000000     17970000.0


## Step 7: Create the Dual-Axis Chart

Visualize the **Average Installs** and **Total Revenue** for the top 3 app categories. The chart is displayed only between **1:00 PM and 2:00 PM IST**.

In [96]:
# ============================================================
# Task 6 - Final Interactive Dual-Axis Chart using Plotly
# ============================================================

from datetime import datetime
import pytz
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Get current IST time
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

# Allowed display time: 1 PM to 2 PM IST
start_time = datetime.strptime("13:00", "%H:%M").time()
end_time = datetime.strptime("14:00", "%H:%M").time()

# Display chart only during allowed time
if start_time <= current_time <= end_time:

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    # Bar chart for Average Installs
    fig.add_trace(
        go.Bar(
            x=task6_summary["Category"],
            y=task6_summary["Average_Installs"],
            name="Average Installs",
            marker_color="steelblue",
            opacity=0.85,
            hovertemplate=
                "<b>Category:</b> %{x}<br>" +
                "<b>Average Installs:</b> %{y:,.0f}<extra></extra>"
        ),
        secondary_y=False
    )

    # Line chart for Revenue
    fig.add_trace(
        go.Scatter(
            x=task6_summary["Category"],
            y=task6_summary["Total_Revenue"],
            name="Revenue",
            mode="lines+markers",
            line=dict(color="crimson", width=4),
            marker=dict(size=10),
            hovertemplate=
                "<b>Category:</b> %{x}<br>" +
                "<b>Revenue:</b> $%{y:,.0f}<extra></extra>"
        ),
        secondary_y=True
    )

    # Layout formatting
    fig.update_layout(
        title=dict(
            text="Dual-Axis Comparison of Average Installs and Revenue<br><sup>Top 3 App Categories</sup>",
            x=0.5,
            font=dict(size=21)
        ),
        xaxis_title="Category",
        template="plotly_white",
        width=1050,
        height=600,
        hovermode="x unified",
        legend=dict(
            x=0.02,
            y=0.98,
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="lightgray",
            borderwidth=1
        ),
        margin=dict(l=80, r=100, t=100, b=80)
    )

    # Left Y-axis
    fig.update_yaxes(
        title_text="Average Installs",
        tickformat=",",
        secondary_y=False
    )

    # Right Y-axis
    fig.update_yaxes(
        title_text="Revenue ($)",
        tickprefix="$",
        tickformat=",",
        secondary_y=True
    )

    fig.show()

else:
    print("Access restricted: This visualization is available only between 1:00 PM and 2:00 PM IST.")

Access restricted: This visualization is available only between 1:00 PM and 2:00 PM IST.


### Note

Revenue was calculated as:

**Revenue = Installs × Price**

After applying the required filter **Revenue ≥ $10,000**, all free applications (Price = $0) were excluded. Therefore, the final visualization contains only paid applications while satisfying all internship requirements.